# Анализ данных

Запускать из корня проекта. Ноутбук использует подготовленные parquet и OOF-предсказания версии v2.


In [1]:
from pathlib import Path
import os

if not Path('main.py').exists() and (Path.cwd().parent / 'main.py').exists():
    os.chdir(Path.cwd().parent)


## Структура диалогов, повторы и сдвиг train/test


In [2]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from catboost import CatBoostClassifier
from scipy.stats import ks_2samp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, train_test_split

from main import FEATURES, row_indices


ROOT = Path.cwd()
OUTPUTS = ROOT / "outputs"


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().casefold())


def rates(values, labels) -> dict:
    groups = defaultdict(list)
    for value, label in zip(values, labels):
        groups[str(value)].append(int(label))
    return {
        name: {"rows": len(items), "bot_rate": round(float(np.mean(items)), 4)}
        for name, items in sorted(groups.items())
    }


def error_slices(values, labels, probabilities) -> dict:
    groups = defaultdict(list)
    for value, label, probability in zip(values, labels, probabilities):
        clipped = np.clip(probability, 1e-8, 1 - 1e-8)
        loss = -(label * np.log(clipped) + (1 - label) * np.log(1 - clipped))
        groups[str(value)].append(float(loss))
    return {
        name: {"rows": len(items), "logloss": round(float(np.mean(items)), 4)}
        for name, items in sorted(groups.items())
    }


def nearest_neighbor_summary(train: pl.DataFrame) -> dict:
    labels = train["is_bot"].to_numpy()
    pair_classes = labels.reshape(-1, 2) @ np.array([2, 1])
    dialogue_indices = np.arange(len(pair_classes))
    train_dialogues, val_dialogues = train_test_split(
        dialogue_indices, test_size=.25, random_state=23, stratify=pair_classes
    )
    tr = row_indices(train_dialogues)
    va = row_indices(val_dialogues)
    texts = [normalize(text) for text in train["text"].to_list()]
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 5), min_df=2, max_features=120000)
    x_train = vectorizer.fit_transform([texts[i] for i in tr])
    x_val = vectorizer.transform([texts[i] for i in va])
    similarity = cosine_similarity(x_val, x_train)
    nearest = similarity.argmax(axis=1)
    closest = similarity[np.arange(len(va)), nearest]
    correct = labels[tr][nearest] == labels[va]
    result = {}
    for low, high in [(0, .2), (.2, .5), (.5, .8), (.8, 1.001)]:
        mask = (closest >= low) & (closest < high)
        if mask.any():
            result[f"{low:.1f}-{min(high, 1):.1f}"] = {
                "rows": int(mask.sum()),
                "nearest_label_accuracy": round(float(correct[mask].mean()), 4),
            }
    return result


def adversarial_auc(train: pl.DataFrame, test: pl.DataFrame) -> float:
    train_dialogues = sorted(train["dialog_id"].unique().to_list())
    test_dialogues = sorted(test["dialog_id"].unique().to_list())
    if set(train_dialogues) & set(test_dialogues):
        raise ValueError("Train and test dialogue IDs overlap")
    groups = train_dialogues + test_dialogues
    source_by_dialogue = {dialog_id: 0 for dialog_id in train_dialogues}
    source_by_dialogue.update({dialog_id: 1 for dialog_id in test_dialogues})
    all_data = pl.concat([train.select(["dialog_id"] + FEATURES), test.select(["dialog_id"] + FEATURES)])
    x = all_data.select(FEATURES).to_numpy().astype(float)
    row_groups = all_data["dialog_id"].to_list()
    source = np.array([source_by_dialogue[dialog_id] for dialog_id in row_groups])
    group_source = np.array([source_by_dialogue[dialog_id] for dialog_id in groups])
    predictions = np.zeros(len(all_data))
    cv = StratifiedKFold(4, shuffle=True, random_state=23)
    for fold, (train_groups, val_groups) in enumerate(cv.split(groups, group_source)):
        train_set = {groups[i] for i in train_groups}
        val_set = {groups[i] for i in val_groups}
        tr = np.array([i for i, dialog_id in enumerate(row_groups) if dialog_id in train_set])
        va = np.array([i for i, dialog_id in enumerate(row_groups) if dialog_id in val_set])
        model = CatBoostClassifier(
            iterations=250, depth=4, learning_rate=.05, loss_function="Logloss",
            random_seed=23 + fold, thread_count=4, verbose=False
        )
        model.fit(x[tr], source[tr])
        predictions[va] = model.predict_proba(x[va])[:, 1]
    return round(float(roc_auc_score(source, predictions)), 4)


def plot_report(report: dict) -> None:
    figure, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors = ["#627e9b", "#739a80", "#c4775d"]
    count_keys = ["-1", "0", "1"]
    count_values = [report["label_rates"]["n_diff"][key] for key in count_keys]
    axes[0].bar(count_keys, [item["bot_rate"] for item in count_values], color=colors)
    axes[0].set_title("Bot rate by message count difference")
    axes[0].set_xlabel("Clipped message count difference")
    axes[0].set_ylabel("Bot rate")
    axes[0].set_ylim(0, 1)
    for x, item in enumerate(count_values):
        axes[0].text(x, item["bot_rate"] + .025, str(item["rows"]), ha="center", fontsize=9)

    length_keys = ["2-4", "5-8", "9+"]
    length_values = [report["error_slices"]["dialogue_length"][key] for key in length_keys]
    axes[1].bar(length_keys, [item["logloss"] for item in length_values], color=colors)
    axes[1].set_title("OOF LogLoss by dialogue length")
    axes[1].set_xlabel("Messages in dialogue")
    axes[1].set_ylabel("LogLoss")
    axes[1].set_ylim(0, .34)
    for x, item in enumerate(length_values):
        axes[1].text(x, item["logloss"] + .008, str(item["rows"]), ha="center", fontsize=9)

    calibration = report["calibration"]
    axes[2].plot([0, 1], [0, 1], linestyle="--", color="#777777", linewidth=1)
    axes[2].plot(
        [item["mean_prediction"] for item in calibration],
        [item["bot_rate"] for item in calibration],
        marker="o", color="#446e9a", linewidth=1.5,
    )
    axes[2].set_title("OOF probability calibration")
    axes[2].set_xlabel("Mean predicted probability")
    axes[2].set_ylabel("Observed bot rate")
    axes[2].set_xlim(0, 1)
    axes[2].set_ylim(0, 1)
    figure.tight_layout()
    figure.savefig(OUTPUTS / "analysis/eda_charts.png", dpi=160)
    plt.close(figure)


def main() -> None:
    (OUTPUTS / "analysis").mkdir(parents=True, exist_ok=True)
    train = pl.read_parquet(OUTPUTS / "v2/processed/train_participants.parquet").sort(["dialog_id", "participant_index"])
    test = pl.read_parquet(OUTPUTS / "v2/processed/test_participants.parquet")
    train_messages = pl.read_parquet(OUTPUTS / "v2/processed/train_messages.parquet")
    oof = pl.read_parquet(OUTPUTS / "v2/oof_predictions.parquet").sort(["dialog_id", "participant_index"])
    labels = train["is_bot"].to_numpy()
    probabilities = oof["blend_pair"].to_numpy()
    train_texts = [normalize(text) for text in train["text"].to_list()]
    test_texts = [normalize(text) for text in test["text"].to_list()]

    by_text = defaultdict(list)
    for text, label in zip(train_texts, labels):
        if text:
            by_text[text].append(int(label))
    repeated = {text: values for text, values in by_text.items() if len(values) > 1}
    with (ROOT / "data/train.json").open(encoding="utf-8") as file:
        raw_train = json.load(file)
    with (ROOT / "data/test.json").open(encoding="utf-8") as file:
        raw_test = json.load(file)
    signature = lambda messages: tuple((item["participant_index"], normalize(item["text"])) for item in messages)
    train_signatures = {signature(messages) for messages in raw_train.values()}
    exact_overlap = sum(signature(messages) in train_signatures for messages in raw_test.values())

    n_diff = np.clip(train["n_diff"].to_numpy(), -1, 1)
    script = np.where(train["cyrillic"].to_numpy() > train["latin"].to_numpy(), "cyrillic", "latin_or_other")
    lengths = train_messages.group_by("dialog_id").len()
    length_by_dialogue = dict(zip(lengths["dialog_id"], lengths["len"]))
    dialogue_length = np.array([length_by_dialogue[dialog_id] for dialog_id in train["dialog_id"]])
    dialogue_bins = np.where(dialogue_length <= 4, "2-4", np.where(dialogue_length <= 8, "5-8", "9+"))
    drift = sorted([
        {
            "feature": name,
            "ks": round(float(ks_2samp(train[name].to_numpy(), test[name].to_numpy()).statistic), 4),
            "train_median": round(float(np.median(train[name].to_numpy())), 4),
            "test_median": round(float(np.median(test[name].to_numpy())), 4),
        }
        for name in FEATURES
    ], key=lambda item: -item["ks"])[:10]

    calibration = []
    for i in range(10):
        low = i / 10
        high = (i + 1) / 10
        mask = (probabilities >= low) & (probabilities < high if i < 9 else probabilities <= high)
        if mask.any():
            calibration.append({
                "bin": f"{low:.1f}-{high:.1f}",
                "rows": int(mask.sum()),
                "mean_prediction": round(float(probabilities[mask].mean()), 4),
                "bot_rate": round(float(labels[mask].mean()), 4),
            })

    model = CatBoostClassifier()
    model.load_model(str(OUTPUTS / "v2/models/catboost_extended.cbm"))
    importance = sorted(zip(FEATURES, model.get_feature_importance()), key=lambda item: -item[1])[:12]
    report = {
        "label_rates": {
            "n_diff": rates(n_diff, labels),
            "first_speaker": rates(train["first_speaker"].to_numpy(), labels),
            "last_speaker": rates(train["is_last_speaker"].to_numpy(), labels),
            "exact_echo": rates(train["exact_echo_share"].to_numpy() > 0, labels),
        },
        "error_slices": {
            "n_diff": error_slices(n_diff, labels, probabilities),
            "dialogue_length": error_slices(dialogue_bins, labels, probabilities),
            "script": error_slices(script, labels, probabilities),
        },
        "repeated_participant_texts": {
            "templates": len(repeated),
            "train_rows": sum(len(values) for values in repeated.values()),
            "label_pure_templates": sum(len(set(values)) == 1 for values in repeated.values()),
            "test_rows_matching_train": sum(text in by_text for text in test_texts),
        },
        "exact_train_test_dialogues": exact_overlap,
        "nearest_neighbor": nearest_neighbor_summary(train),
        "train_test_adversarial_auc": adversarial_auc(train, test),
        "largest_train_test_ks": drift,
        "calibration": calibration,
        "top_catboost_features": [
            {"feature": name, "importance": round(float(value), 3)}
            for name, value in importance
        ],
    }
    (OUTPUTS / "analysis/eda_extended.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    plot_report(report)

    print(json.dumps({
        "adversarial_auc": report["train_test_adversarial_auc"],
        "exact_train_test_dialogues": exact_overlap,
        "top_features": report["top_catboost_features"][:5],
    }, ensure_ascii=False, indent=2))

main()


{
  "adversarial_auc": 0.5198,
  "exact_train_test_dialogues": 3,
  "top_features": [
    {
      "feature": "is_last_speaker",
      "importance": 11.901
    },
    {
      "feature": "n_diff",
      "importance": 7.387
    },
    {
      "feature": "avg_prev_overlap",
      "importance": 5.611
    },
    {
      "feature": "uppercase",
      "importance": 5.359
    },
    {
      "feature": "shared_vocab_ratio",
      "importance": 4.703
    }
  ]
}


![Структура диалогов и ошибки](../outputs/analysis/eda_charts.png)

Когда участник написал меньше реплик, доля ботов составила 4,5%. При равном числе реплик она была 37,1%, при большем - 80,9%. Среди авторов последней реплики ботов 73,0%, среди остальных 8,6%. Точный повтор предыдущей реплики собеседника связан с долей ботов 79,8% против 30,7% без повтора.

Полные тексты участников повторялись в train 168 разными шаблонами, но только 47 шаблонов всегда имели одну метку. Проверка train/test по числовым признакам дала ROC AUC 0,520. Она не исключает сдвиг в самих текстах или метках. На диалогах из 2-4 реплик OOF logloss около 0,289, на 9 и более - 0,074.


## Длина отдельных реплик


In [3]:
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np


root = Path.cwd()
dialogues = json.loads((root / "data" / "train.json").read_text(encoding="utf-8"))
lengths = np.array([len(message["text"]) for dialogue in dialogues.values() for message in dialogue])
shown = lengths[lengths <= 100]
median = float(np.median(lengths))

fig, ax = plt.subplots(figsize=(11, 5.4))
fig.patch.set_facecolor("white")
ax.hist(shown, bins=np.arange(0, 106, 5), color="#7055F4", edgecolor="white", linewidth=1)
ax.axvline(median, color="#1876AF", linewidth=2.5, linestyle="--", label=f"Медиана: {median:.0f} символов")
ax.set_title("Длина реплик в train", loc="left", fontsize=18, weight="bold", pad=16)
ax.set_xlabel("Длина реплики, символы")
ax.set_ylabel("Число реплик")
ax.set_xlim(0, 100)
ax.grid(axis="y", color="#E8E8EE", linewidth=1)
ax.set_axisbelow(True)
ax.legend(frameon=False)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#D1D1DA")

hidden = len(lengths) - len(shown)
count = f"{len(lengths):_}".replace("_", " ")
fig.text(0.10, 0.035, f"Всего {count} реплик. На графике длины до 100 символов, длиннее - {hidden} ({hidden / len(lengths):.1%}).",
         color="#65616F", fontsize=10)
fig.subplots_adjust(left=0.10, right=0.97, top=0.88, bottom=0.21)
output = root / 'defense' / ("Длина реплик - Python.png")
fig.savefig(output, dpi=200)
plt.close(fig)
print(output)


C:\Users\Mayya\Documents\ЦУ\Production ML\project\defense\Длина реплик - Python.png


![Длина реплик](../defense/Длина%20реплик%20-%20Python.png)

Медиана длины реплики равна 8 символам. На графике показаны длины до 100 символов, оставшиеся длинные реплики указаны в подписи.


## Как менялось качество


In [4]:
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter


root = Path.cwd()
versions = ["v1", "v2", "seed42", "v3"]
runs = [9348, 9349, 9350, 9351]
public = [0.2356, 0.1407, 0.1407, 0.1385]
local = [
    json.loads((root / "outputs" / version / "results.json").read_text(encoding="utf-8"))["oof_logloss"]["blend_pair"]
    for version in versions
]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("white")
x = list(range(len(versions)))
ax.plot(x, local, color="#7055F4", marker="o", linewidth=2.8, markersize=8, label="OOF, локально")
ax.plot(x, public, color="#1876AF", marker="o", linewidth=2.8, markersize=8, label="Public, платформа")

for points, offset in ((local, 10), (public, -22)):
    for position, value in zip(x, points):
        ax.annotate(f"{value:.4f}".replace(".", ","), (position, value), xytext=(0, offset),
                    textcoords="offset points", ha="center", fontsize=10)

ax.set_xticks(x, [f"{version}\n#{run}" for version, run in zip(versions, runs)])
ax.set_ylim(0.11, 0.275)
ax.set_yticks([0.12, 0.16, 0.20, 0.24])
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.set_ylabel("LogLoss (меньше лучше)")
ax.set_title("Как менялось качество модели", loc="left", fontsize=18, weight="bold", pad=22)
ax.grid(axis="y", color="#E8E8EE", linewidth=1)
ax.set_axisbelow(True)
ax.legend(loc="upper right", frameon=False)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#D1D1DA")

fig.text(0.10, 0.07, "Соответствие версий и старых загрузок предварительное. Исходные CSV с платформы не сверены.",
         color="#65616F", fontsize=10)
fig.subplots_adjust(left=0.10, right=0.96, top=0.83, bottom=0.22)
output = root / 'defense' / ("Динамика качества - Python.png")
fig.savefig(output, dpi=200)
plt.close(fig)
print(output)


C:\Users\Mayya\Documents\ЦУ\Production ML\project\defense\Динамика качества - Python.png


![Качество моделей](../defense/Динамика%20качества%20-%20Python.png)

OOF и public LogLoss посчитаны на разных выборках. Соответствие старых отправок локальным версиям предварительное, потому что на платформу файлы загружали с одинаковым именем.
